# Food Craft — Unsloth Llama-3.2-11B-Vision Fine-Tuning & Hugging Face Push

**Works on Kaggle Notebooks (`GPU T4 x2`) and Google Colab (`T4 GPU`)!**
1. Select a **GPU runtime** (`Accelerator -> GPU T4 x2` + `Internet -> ON` on Kaggle, or `T4 GPU` on Colab).
2. Add `HF_TOKEN` (with **Write** access under `pavankumarpalla`) to Secrets.


In [ ]:
import os
for loader in ('colab', 'kaggle'):
    try:
        if loader == 'colab':
            from google.colab import userdata
            tok = userdata.get('HF_TOKEN')
        else:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret('HF_TOKEN')
        if tok:
            os.environ['HF_TOKEN'] = tok.strip()
            break
    except Exception:
        pass
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN with Write permissions in Colab/Kaggle Secrets before running!'
!nvidia-smi

In [ ]:
import os
if os.path.exists('.git'):
    !git pull
elif os.path.exists('FoodCraft/.git'):
    %cd FoodCraft
    !git pull
else:
    !git clone https://github.com/PavanKumarPalla/FoodCraft.git
    %cd FoodCraft
!rm -rf data/raw_datasets data/unsloth_foodcraft

In [ ]:
!python finetune_workspace/install_deps.py

In [ ]:
# Verify GPU VRAM, HF_TOKEN write permissions, and disk space BEFORE training
!python finetune_workspace/train_and_push.py --preflight-only --push-merged

In [ ]:
# Stream Food Craft datasets (~1.5 GB total disk, zero 19GB quota overflow) + index pavankumarpalla/FoodCraft
!python finetune_workspace/prepare_dataset.py --stream-cloud --prune-raw --max-per-source 4000

In [ ]:
# Fine-tune Llama-3.2-11B-Vision (~1.5-2h on single-tile 560px) and push LoRA + merged model to HF Hub
!python finetune_workspace/train_and_push.py --max-steps 150 --grad-accum 4 --push-merged